In [77]:
from pathlib import Path
import importlib
import copy
import json
import tqdm

import os
import numpy as np
import pandas as pd
import torch

import matplotlib.pyplot as plt

from src.datasets import (
    prepare_data,
    split_dataset,
)
import os.path as osp

In [42]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# device = 'cpu'

In [6]:
exp_dir_path = Path("out_Termo_Ablation_heads/8_heads")
out_dir_path = osp.join(exp_dir_path, 'results')

In [ ]:
with open(Path(exp_dir_path) / 'params.json', 'r') as f:
    cfg = json.load(f)
pair_dataset, pair_scalers , _, _, test_loader, pair_ideal = prepare_data(
    cfg['dataset'],
    cfg['dataloader'],
    cfg['utils']['seed']
)
cfg['dataset']['load'] = True

Датасет загружен из файла: data_Termo_Heat.pt
Готово! Количество графов: 42525, идеальных графов: 2
Train: 29767, Val: 6378, Test: 6380


In [79]:
train_dataset, val_dataset, test_dataset = split_dataset(
    pair_dataset,
    cfg['dataloader']['train_ratio'],
    cfg['dataloader']['val_ratio'],
    seed=cfg['utils']['seed']
)

In [57]:
print(len(pair_dataset))
print(pair_dataset[0])

42525
(Data(x=[107, 9], edge_index=[2, 107], edge_attr=[107, 5], global_attrs=[1, 5], edge_label=[1], edge_moded=[107, 1], nodes_fp='datasets/Termo_model_fwd/Tout_-5/problem/tube_119/Thermo_model_db_n_nodes_pr2_fwd.csv', edges_fp='datasets/Termo_model_fwd/Tout_-5/problem/tube_119/Thermo_model_db_n_tubes_pr2_fwd.csv'), Data(x=[107, 9], edge_index=[2, 107], edge_attr=[107, 5], global_attrs=[1, 5], edge_label=[1], edge_moded=[107, 1], nodes_fp='datasets/Termo_model_bwd/Tout_-5/problem/tube_119/Thermo_model_db_n_nodes_pr2_bwd.csv', edges_fp='datasets/Termo_model_bwd/Tout_-5/problem/tube_119/Thermo_model_db_n_tubes_pr2_bwd.csv'))


In [10]:
dataset= pair_dataset[0]
scalers = pair_scalers[0]
ideal_dataset = pair_ideal[0]

In [43]:
in_node_dim = dataset[0].x.shape[1]
in_edge_dim = dataset[0].edge_attr.shape[1]
model_module = importlib.import_module(f"src.models.{cfg['model']['name']}")
ModelClass = getattr(model_module, cfg['model']['name'])

def create_model():
    return ModelClass(
        in_node_dim=in_node_dim,
        in_edge_dim=in_edge_dim,
        **cfg['model']['kwargs']
    )

model = create_model().to(device)

state = torch.load(exp_dir_path / 'best_model.pth', weights_only=True)
model.load_state_dict(state)
model.eval()



EdgeClassifierNetwork_Attr(
  (node_encoder): NodeEncoder(
    (convs): ModuleList(
      (0): GATv2Conv(14, 128, heads=1)
      (1-7): 7 x GATv2Conv(128, 128, heads=1)
    )
    (norms): ModuleList(
      (0-7): 8 x LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
    (jump): JumpingKnowledge(cat)
  )
  (edge_init): Sequential(
    (0): Linear(in_features=2058, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=128, bias=True)
  )
  (edge_update_layers): ModuleList(
    (0-7): 8 x EdgeAttentionLayerFast(
      (q_proj): Linear(in_features=128, out_features=128, bias=True)
      (k_proj): Linear(in_features=128, out_features=128, bias=True)
      (v_proj): Linear(in_features=128, out_features=128, bias=True)
      (out_proj): Linear(in_features=128, out_features=128, bias=True)
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
    )


In [ ]:
from torch_geometric.explain import Explainer, GNNExplainer, AttentionExplainer, GraphMaskExplainer, PGExplainer, CaptumExplainer
import torch_geometric

In [46]:
class ModelWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x, edge_index, data):
        return self.model(x, edge_index, torch_geometric.data.Batch.from_data_list([data]))

In [47]:
wrapped = ModelWrapper(model)

In [15]:
dataset[0].to(device)

Data(x=[107, 9], edge_index=[2, 107], edge_attr=[107, 5], global_attrs=[1, 5], edge_label=[1], edge_moded=[107, 1], nodes_fp='datasets/Termo_model_fwd/Tout_-5/problem/tube_119/Thermo_model_db_n_nodes_pr2_fwd.csv', edges_fp='datasets/Termo_model_fwd/Tout_-5/problem/tube_119/Thermo_model_db_n_tubes_pr2_fwd.csv')

In [21]:
len(pair_dataset)

42525

In [ ]:
def interpretation(model, dataset, fwd_or_bwd, node_mask_type = 'object', edge_mask_type = 'object'):
    '''
    Generate GNNExplainer explanations for forward (0) or backward (1) graph.
    
    Parameters
    ----------
    model : torch.nn.Module
        Trained GNN model for graph-level multiclass classification.
    dataset : dict
        Container with 'fwd' and 'bwd' graph data objects.
    fwd_or_bwd : int
        0 for forward graph, 1 for backward graph.
    node_mask_type : str or None, optional
        Type of node mask: None, 'object', 'attributes', or 'common_attributes'.
        Default 'object'.
    edge_mask_type : str or None, optional
        Type of edge mask: None or 'object'. Default 'object'.
    
    Returns
    -------
    explanation : Explanation object
        Contains node_mask and edge_mask with importance scores.
    '''
    
    explainer = Explainer(
        model=model,
        algorithm=GNNExplainer(),
        explanation_type='model',          # Объясняем предсказание модели
        node_mask_type=node_mask_type,       # Хотим понять важность признаков узлов
        edge_mask_type=edge_mask_type,           # Хотим понять важность ребер
        model_config=dict(
            mode='multiclass_classification',
            task_level='graph',
            return_type='log_probs',
        ),
    )
    dataset[fwd_or_bwd].to(device)
    explanation = explainer(dataset[fwd_or_bwd].x, dataset[fwd_or_bwd].edge_index, index=None, data=dataset[fwd_or_bwd])
    return explanation

In [75]:
explanation = interpretation(wrapped, pair_dataset[4242], 0, node_mask_type='object', edge_mask_type='object') 

In [76]:
print(explanation.node_mask)  # Важность узлов и их признаков
print(explanation.edge_mask)  # Важность ребер


tensor([[0.2763],
        [0.3005],
        [0.3034],
        [0.5320],
        [0.7043],
        [0.6229],
        [0.5507],
        [0.4026],
        [0.4956],
        [0.2895],
        [0.6182],
        [0.5307],
        [0.5217],
        [0.5496],
        [0.3689],
        [0.6279],
        [0.3973],
        [0.7536],
        [0.5664],
        [0.6611],
        [0.7025],
        [0.7777],
        [0.5171],
        [0.4253],
        [0.5485],
        [0.3684],
        [0.5528],
        [0.5806],
        [0.3975],
        [0.4935],
        [0.5300],
        [0.5221],
        [0.3966],
        [0.3498],
        [0.3480],
        [0.3029],
        [0.3806],
        [0.5558],
        [0.6488],
        [0.4011],
        [0.4857],
        [0.4923],
        [0.4489],
        [0.5448],
        [0.5466],
        [0.5724],
        [0.6012],
        [0.5896],
        [0.6157],
        [0.5297],
        [0.7240],
        [0.7883],
        [0.5629],
        [0.2973],
        [0.5406],
        [0

In [80]:
explanation = interpretation(wrapped, test_dataset[42], 0, node_mask_type='object', edge_mask_type='object') 

In [82]:
print(explanation.node_mask)
print(explanation.edge_mask)

tensor([[0.3039],
        [0.3341],
        [0.3212],
        [0.6982],
        [0.3277],
        [0.6973],
        [0.6607],
        [0.5805],
        [0.3877],
        [0.6289],
        [0.5011],
        [0.5360],
        [0.5510],
        [0.2835],
        [0.2810],
        [0.3122],
        [0.2843],
        [0.3876],
        [0.3277],
        [0.3680],
        [0.2844],
        [0.3472],
        [0.3177],
        [0.2861],
        [0.2878],
        [0.2848],
        [0.3390],
        [0.3657],
        [0.5090],
        [0.5443],
        [0.5408],
        [0.2876],
        [0.4832],
        [0.5403],
        [0.7079],
        [0.5540],
        [0.2633],
        [0.6611],
        [0.6879],
        [0.2555],
        [0.3707],
        [0.4826],
        [0.5288],
        [0.2947],
        [0.3923],
        [0.3649],
        [0.2952],
        [0.3207],
        [0.3090],
        [0.2904],
        [0.6731],
        [0.4337],
        [0.5620],
        [0.6389],
        [0.3331],
        [0

In [67]:
batch = next(iter(test_loader))

In [68]:
batch

[DataBatch(x=[1712, 9], edge_index=[2, 1712], edge_attr=[1712, 5], global_attrs=[16, 5], edge_label=[16], edge_moded=[1712, 1], nodes_fp=[16], edges_fp=[16], batch=[1712], ptr=[17]),
 DataBatch(x=[1712, 9], edge_index=[2, 1712], edge_attr=[1712, 5], global_attrs=[16, 5], edge_label=[16], edge_moded=[1712, 1], nodes_fp=[16], edges_fp=[16], batch=[1712], ptr=[17])]

In [ ]:
explanation = interpretation(wrapped, batch, 0, node_mask_type='common_attributes', edge_mask_type='object') # Можно и батчи целые подавать (ОДНАКО ТАМ БУДУТ ПОПАДАТЬСЯ РАЗНЫЕ КЛАССЫ)

In [72]:
node_attr = ['pos_x', 'pos_y', 'types_def', 'types_usr', 'types_src', 'P', 'Temp', 'P_ideal', 'Temp_ideal']  # Атрибуты узлов
print(explanation.node_mask)  # Важность узлов и их признаков

tensor([[0.4077, 0.5661, 0.3666, 0.2999, 0.0000, 0.4849, 0.4821, 0.4011, 0.5149]],
       device='cuda:0')


In [73]:
print(explanation.edge_mask)  # Важность ребер


tensor([0.2774, 0.2872, 0.3052,  ..., 0.2788, 0.2812, 0.2755], device='cuda:0')


In [ ]:
edge_attr = ['d', 'l', 'Vid_fwd', 'Vid_bwd', 'Vid_usr']  # Атрибуты ребер